# 🔧 Notebook 2: Preprocessing Pipeline

## Grayscale Conversion → Resize to 10×10 → Row-wise Mean → 1D Signal

---

### Pipeline Stage
```
┌───────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│ MRI Brain    │--->│  Grayscale   │--->│ Resize to    │--->│  Row-wise    │
│ Image (4ch)  │    │  Conversion  │    │   10 × 10    │    │  Mean→1D    │
└───────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
(240×240×4)       (240×240×1)       (10×10)             (10,)
```

### Objective
This notebook implements the first three preprocessing steps of the pipeline:
1. **Grayscale Conversion** - Convert multi-modal MRI (4 channels) to single-channel grayscale
2. **Spatial Resizing** - Downscale from 240×240 to 10×10 pixels
3. **1D Signal Extraction** - Compute row-wise mean to obtain a 10-element 1D signal

The output is a dataset of 1D signals (one per MRI slice) with corresponding labels.

## 1. Import Dependencies

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.transform import resize
from glob import glob
from tqdm import tqdm
import warnings

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    HAS_NIBABEL = False

try:
    import h5py
    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("✅ All imports successful")
print(f"   nibabel: {'available' if HAS_NIBABEL else 'not found'}")
print(f"   h5py   : {'available' if HAS_H5PY else 'not found'}")

## 2. Load Configuration from Notebook 01

In [ ]:
# ============================================================
#  ⚙️  CONFIGURATION
# ============================================================

OUTPUT_DIR = './processed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Try loading config from Notebook 01
config_path = os.path.join(OUTPUT_DIR, 'config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = json.load(f)
    DATA_DIR = config['DATA_DIR']
    USE_SYNTHETIC = config['USE_SYNTHETIC']
    print(f"✅ Loaded config from Notebook 01")
else:
    # Manual configuration
    DATA_DIR = './data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
    USE_SYNTHETIC = not os.path.exists(DATA_DIR)
    print("⚠️ Config not found - using defaults")

# Preprocessing parameters
TARGET_SIZE = (10, 10)       # Resize target
SLICES_PER_SUBJECT = 20     # Number of slices to extract per subject
MAX_SUBJECTS = 50           # Max subjects to process (None for all)

print(f"   Data dir      : {DATA_DIR}")
print(f"   Output dir    : {OUTPUT_DIR}")
print(f"   Synthetic mode: {USE_SYNTHETIC}")
print(f"   Target size   : {TARGET_SIZE}")
print(f"   Slices/subject: {SLICES_PER_SUBJECT}")

## 3. Load Subject Labels

In [ ]:
# Load label mapping
labels_path = os.path.join(OUTPUT_DIR, 'subject_labels.csv')
if os.path.exists(labels_path):
    label_df = pd.read_csv(labels_path)
    label_map = dict(zip(label_df['subject_id'], label_df['label']))
    print(f"✅ Loaded {len(label_map)} subject labels")
    print(f"   Malignant (HGG): {sum(v == 1 for v in label_map.values())}")
    print(f"   Benign (LGG)   : {sum(v == 0 for v in label_map.values())}")
else:
    print("⚠️ Labels file not found. Run Notebook 01 first, or labels will be inferred.")
    label_map = {}

## 4. Preprocessing Functions

### Step 1: Grayscale Conversion

The BraTS dataset has 4 MRI modalities (FLAIR, T1, T1ce, T2). We convert them to a single grayscale channel by computing the **weighted mean** across modalities:

$$I_{gray}(x, y) = \frac{1}{N} \sum_{m=1}^{N} w_m \cdot I_m(x, y)$$

where $N=4$ modalities and $w_m$ are the weights (uniform by default).

In [ ]:
def to_grayscale(multi_modal_slice, weights=None):
    """
    Convert multi-modal MRI slice to single-channel grayscale.
    
    Parameters:
        multi_modal_slice: numpy array of shape (H, W, C) where C = number of modalities
        weights: optional weights for each modality (default: uniform)
    
    Returns:
        Grayscale image of shape (H, W), normalized to [0, 1]
    """
    if multi_modal_slice.ndim == 2:
        # Already grayscale
        gray = multi_modal_slice.copy()
    elif multi_modal_slice.ndim == 3:
        C = multi_modal_slice.shape[2]
        if weights is None:
            weights = np.ones(C) / C
        else:
            weights = np.array(weights) / np.sum(weights)
        
        gray = np.sum(multi_modal_slice * weights[np.newaxis, np.newaxis, :], axis=2)
    else:
        raise ValueError(f"Expected 2D or 3D array, got shape {multi_modal_slice.shape}")
    
    # Normalize to [0, 1]
    if gray.max() > gray.min():
        gray = (gray - gray.min()) / (gray.max() - gray.min())
    
    return gray.astype(np.float32)


print("✅ Grayscale conversion function defined")
print("   Formula: I_gray = (1/N) * Σ(w_m * I_m)")

### Step 2: Resize to 10×10

Aggressively downscale the image from 240×240 to 10×10 pixels using bilinear interpolation. This extreme compression captures only the **global intensity pattern** of the brain, discarding fine structural details while preserving macro-level signal characteristics useful for differential modeling.

In [ ]:
def resize_image(image, target_size=(10, 10)):
    """
    Resize an image to the target dimensions using anti-aliased interpolation.
    
    Parameters:
        image: numpy array of shape (H, W)
        target_size: tuple (target_H, target_W)
    
    Returns:
        Resized image of shape (target_H, target_W)
    """
    resized = resize(image, target_size, anti_aliasing=True, preserve_range=True)
    return resized.astype(np.float32)


print("✅ Resize function defined")
print(f"   Target size: {TARGET_SIZE}")
print(f"   Compression ratio: {240*240}/{TARGET_SIZE[0]*TARGET_SIZE[1]} = {(240*240)/(TARGET_SIZE[0]*TARGET_SIZE[1]):.0f}x")

### Step 3: Row-wise Mean → 1D Signal

Compute the mean intensity along each row to obtain a **10-element 1D signal**:

$$S(i) = \frac{1}{W} \sum_{j=1}^{W} I_{resized}(i, j), \quad i = 1, 2, \ldots, 10$$

This 1D signal captures the **vertical intensity profile** of the brain slice, which differs between benign and malignant tumors due to varying enhancement patterns, edema extent, and tissue disruption.

In [ ]:
def compute_rowwise_mean(image_2d):
    """
    Compute the row-wise mean to convert a 2D image into a 1D signal.
    
    Parameters:
        image_2d: numpy array of shape (H, W)
    
    Returns:
        1D signal of shape (H,)
    """
    signal = np.mean(image_2d, axis=1)  # Mean across columns for each row
    return signal.astype(np.float32)


print("✅ Row-wise mean function defined")
print("   Input:  (10, 10) image")
print("   Output: (10,) 1D signal")

### Complete Preprocessing Pipeline

In [ ]:
def preprocess_slice(multi_modal_slice, target_size=(10, 10)):
    """
    Complete preprocessing pipeline for a single MRI slice.
    
    Pipeline: Multi-modal (H,W,C) → Grayscale (H,W) → Resize (10,10) → Row-mean (10,)
    
    Parameters:
        multi_modal_slice: numpy array of shape (H, W, C) or (H, W)
        target_size: resize target (default: (10, 10))
    
    Returns:
        dict with intermediate results:
            'original': input slice
            'grayscale': grayscale image (H, W)
            'resized': resized image (10, 10)
            'signal_1d': 1D signal (10,)
    """
    # Step 1: Grayscale conversion
    grayscale = to_grayscale(multi_modal_slice)
    
    # Step 2: Resize
    resized = resize_image(grayscale, target_size)
    
    # Step 3: Row-wise mean
    signal_1d = compute_rowwise_mean(resized)
    
    return {
        'original': multi_modal_slice,
        'grayscale': grayscale,
        'resized': resized,
        'signal_1d': signal_1d
    }


print("✅ Complete preprocessing pipeline defined")

## 5. Visualize Preprocessing Steps

Demonstrate the preprocessing pipeline on sample slices, showing each transformation step.

In [ ]:
def generate_demo_slices(n_samples=4):
    """
    Generate synthetic MRI-like slices for demonstration.
    Returns list of (multi_modal_slice, label, name) tuples.
    """
    samples = []
    H, W = 240, 240
    y, x = np.ogrid[-1:1:H*1j, -1:1:W*1j]
    brain_mask = (x**2 / 0.7**2 + y**2 / 0.9**2) < 1
    
    for i in range(n_samples):
        is_malignant = i % 2 == 0
        base = np.random.uniform(0.3, 0.6)
        channels = []
        
        for mod_scale in [1.0, 0.8, 0.9, 1.1]:
            ch = np.zeros((H, W), dtype=np.float32)
            ch[brain_mask] = base * mod_scale + np.random.normal(0, 0.04, brain_mask.sum())
            
            # Add tumor
            tcx, tcy = np.random.randint(90, 150, 2)
            tr = 30 if is_malignant else 15
            tumor = ((np.arange(W) - tcx)**2 + (np.arange(H)[:, None] - tcy)**2) < tr**2
            tumor = tumor & brain_mask
            enhancement = 1.4 if is_malignant else 1.15
            ch[tumor] *= enhancement
            
            channels.append(np.clip(ch, 0, 1))
        
        multi_modal = np.stack(channels, axis=-1)  # (240, 240, 4)
        label = 1 if is_malignant else 0
        name = f"{'Malignant' if is_malignant else 'Benign'} #{i}"
        samples.append((multi_modal, label, name))
    
    return samples


# Generate or load demo slices
if USE_SYNTHETIC:
    demo_samples = generate_demo_slices(4)
    print(f"✅ Generated {len(demo_samples)} demo slices")
else:
    # Load real slices from first few subjects
    demo_samples = []
    subject_dirs_list = sorted(glob(os.path.join(DATA_DIR, 'BraTS20_Training_*')))
    
    for subj_dir in subject_dirs_list[:4]:
        subj_id = os.path.basename(subj_dir)
        modalities = []
        for mod in ['flair', 't1', 't1ce', 't2']:
            nii_path = os.path.join(subj_dir, f'{subj_id}_{mod}.nii')
            nii_gz_path = os.path.join(subj_dir, f'{subj_id}_{mod}.nii.gz')
            if os.path.exists(nii_path):
                vol = nib.load(nii_path).get_fdata()
            elif os.path.exists(nii_gz_path):
                vol = nib.load(nii_gz_path).get_fdata()
            else:
                vol = np.zeros((240, 240, 155))
            modalities.append(vol[:, :, 77])  # Middle slice
        
        multi_modal = np.stack(modalities, axis=-1).astype(np.float32)
        # Normalize
        if multi_modal.max() > 0:
            multi_modal /= multi_modal.max()
        
        label = label_map.get(subj_id, 0)
        name = f"{subj_id} ({'Malig.' if label else 'Benign'})"
        demo_samples.append((multi_modal, label, name))
    
    print(f"✅ Loaded {len(demo_samples)} real slices for demo")

In [ ]:
def visualize_pipeline(samples, target_size=(10, 10)):
    """
    Visualize the complete preprocessing pipeline step by step.
    """
    n = len(samples)
    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    
    fig.suptitle('Preprocessing Pipeline Visualization', fontsize=16, fontweight='bold', y=1.01)
    
    col_titles = ['Original (4 modalities)', 'Grayscale', 'Resized (10×10)', 
                  '10×10 Heatmap', '1D Signal (Row-mean)']
    
    for row, (multi_modal, label, name) in enumerate(samples):
        result = preprocess_slice(multi_modal, target_size)
        
        # Column 0: Original (show composite)
        if multi_modal.ndim == 3:
            # Show first 3 channels as RGB-like composite
            composite = multi_modal[:, :, :3]
            composite = composite / (composite.max() + 1e-8)
            axes[row, 0].imshow(composite)
        else:
            axes[row, 0].imshow(multi_modal, cmap='gray')
        axes[row, 0].set_ylabel(name, fontsize=11, fontweight='bold', color='darkred' if label else 'darkgreen')
        
        # Column 1: Grayscale
        axes[row, 1].imshow(result['grayscale'], cmap='gray')
        axes[row, 1].set_title(f"Shape: {result['grayscale'].shape}" if row == 0 else '', fontsize=9)
        
        # Column 2: Resized
        axes[row, 2].imshow(result['resized'], cmap='gray', interpolation='nearest')
        
        # Column 3: Resized heatmap (annotated)
        im = axes[row, 3].imshow(result['resized'], cmap='viridis', interpolation='nearest')
        # Annotate each cell with its value
        for i in range(target_size[0]):
            for j in range(target_size[1]):
                val = result['resized'][i, j]
                color = 'white' if val < 0.5 else 'black'
                axes[row, 3].text(j, i, f'{val:.2f}', ha='center', va='center', 
                                 fontsize=6, color=color, fontweight='bold')
        
        # Column 4: 1D Signal
        t = np.arange(len(result['signal_1d']))
        color = 'crimson' if label else 'seagreen'
        axes[row, 4].plot(t, result['signal_1d'], 'o-', color=color, linewidth=2, markersize=6)
        axes[row, 4].fill_between(t, result['signal_1d'], alpha=0.2, color=color)
        axes[row, 4].set_xlabel('Row Index')
        axes[row, 4].set_ylabel('Mean Intensity')
        axes[row, 4].set_xlim(-0.5, 9.5)
        axes[row, 4].grid(True, alpha=0.3)
        
        # Remove ticks for image columns
        for c in range(4):
            axes[row, c].set_xticks([])
            axes[row, c].set_yticks([])
    
    # Set column titles
    for c, title in enumerate(col_titles):
        axes[0, c].set_title(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'preprocessing_pipeline_visualization.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n💾 Saved visualization to: {os.path.join(OUTPUT_DIR, 'preprocessing_pipeline_visualization.png')}")


visualize_pipeline(demo_samples, TARGET_SIZE)

## 6. Compare 1D Signals: Benign vs Malignant

Visualize how the 1D row-wise mean signals differ between benign (LGG) and malignant (HGG) tumors.

In [ ]:
# Extract signals from demo samples
benign_signals = []
malignant_signals = []

for multi_modal, label, _ in demo_samples:
    result = preprocess_slice(multi_modal, TARGET_SIZE)
    if label == 0:
        benign_signals.append(result['signal_1d'])
    else:
        malignant_signals.append(result['signal_1d'])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
t = np.arange(10)

# Plot Benign signals
for i, sig in enumerate(benign_signals):
    ax1.plot(t, sig, 'o-', color='seagreen', alpha=0.6, linewidth=1.5, label=f'Benign #{i}')
ax1.set_title('Benign (LGG) Signals', fontsize=13, fontweight='bold', color='seagreen')
ax1.set_xlabel('Row Index (t)')
ax1.set_ylabel('Mean Intensity')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)

# Plot Malignant signals
for i, sig in enumerate(malignant_signals):
    ax2.plot(t, sig, 's-', color='crimson', alpha=0.6, linewidth=1.5, label=f'Malignant #{i}')
ax2.set_title('Malignant (HGG) Signals', fontsize=13, fontweight='bold', color='crimson')
ax2.set_xlabel('Row Index (t)')
ax2.set_ylabel('Mean Intensity')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

# Overlay comparison
for sig in benign_signals:
    ax3.plot(t, sig, 'o-', color='seagreen', alpha=0.5, linewidth=1.5)
for sig in malignant_signals:
    ax3.plot(t, sig, 's-', color='crimson', alpha=0.5, linewidth=1.5)

# Plot group means
if benign_signals:
    mean_b = np.mean(benign_signals, axis=0)
    ax3.plot(t, mean_b, 'o-', color='darkgreen', linewidth=3, label='Benign (mean)', markersize=8)
if malignant_signals:
    mean_m = np.mean(malignant_signals, axis=0)
    ax3.plot(t, mean_m, 's-', color='darkred', linewidth=3, label='Malignant (mean)', markersize=8)

ax3.set_title('Comparison: Benign vs Malignant', fontsize=13, fontweight='bold')
ax3.set_xlabel('Row Index (t)')
ax3.set_ylabel('Mean Intensity')
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. Process Entire Dataset

Apply the preprocessing pipeline to all subjects in the dataset. For each subject:
1. Load all 4 modalities
2. Select representative slices (around the brain center where tumors are most likely)
3. Apply the full pipeline to each slice
4. Store the resulting 1D signals with labels

In [ ]:
def process_dataset_nifti(data_dir, label_map, target_size=(10, 10),
                          slices_per_subject=20, max_subjects=None):
    """
    Process the entire BraTS dataset from NIfTI format.
    
    Returns:
        signals: numpy array of shape (N, signal_length)
        labels: numpy array of shape (N,)
        metadata: list of dicts with subject_id and slice_no
    """
    subject_dirs = sorted(glob(os.path.join(data_dir, 'BraTS20_Training_*')))
    if max_subjects is not None:
        subject_dirs = subject_dirs[:max_subjects]
    
    all_signals = []
    all_labels = []
    all_metadata = []
    
    for subj_dir in tqdm(subject_dirs, desc="Processing subjects"):
        subj_id = os.path.basename(subj_dir)
        label = label_map.get(subj_id, -1)
        if label == -1:
            continue  # Skip subjects without labels
        
        # Load all modalities
        modality_vols = []
        for mod in ['flair', 't1', 't1ce', 't2']:
            nii_path = os.path.join(subj_dir, f'{subj_id}_{mod}.nii')
            nii_gz_path = os.path.join(subj_dir, f'{subj_id}_{mod}.nii.gz')
            
            if os.path.exists(nii_path):
                vol = nib.load(nii_path).get_fdata().astype(np.float32)
            elif os.path.exists(nii_gz_path):
                vol = nib.load(nii_gz_path).get_fdata().astype(np.float32)
            else:
                vol = None
                break
            modality_vols.append(vol)
        
        if vol is None or len(modality_vols) != 4:
            continue
        
        # Normalize each modality
        for i in range(len(modality_vols)):
            vmax = modality_vols[i].max()
            if vmax > 0:
                modality_vols[i] /= vmax
        
        # Select slices around the center (where tumor is most likely)
        total_slices = modality_vols[0].shape[2]
        center = total_slices // 2
        half_range = slices_per_subject // 2
        slice_indices = range(max(0, center - half_range), min(total_slices, center + half_range))
        
        for s_idx in slice_indices:
            # Stack modalities for this slice: (H, W, 4)
            multi_modal = np.stack([v[:, :, s_idx] for v in modality_vols], axis=-1)
            
            # Skip empty slices (very low brain content)
            if np.mean(multi_modal > 0.01) < 0.05:
                continue
            
            # Apply preprocessing pipeline
            result = preprocess_slice(multi_modal, target_size)
            
            all_signals.append(result['signal_1d'])
            all_labels.append(label)
            all_metadata.append({'subject_id': subj_id, 'slice_no': s_idx})
    
    return np.array(all_signals), np.array(all_labels), all_metadata


def process_dataset_synthetic(n_subjects=50, slices_per_subject=20, target_size=(10, 10)):
    """
    Generate and process synthetic BraTS-like data.
    """
    all_signals = []
    all_labels = []
    all_metadata = []
    
    H, W = 240, 240
    y_grid, x_grid = np.ogrid[-1:1:H*1j, -1:1:W*1j]
    brain_mask = (x_grid**2 / 0.7**2 + y_grid**2 / 0.9**2) < 1
    
    for subj_idx in tqdm(range(n_subjects), desc="Processing synthetic subjects"):
        is_hgg = np.random.random() < 0.75
        label = 1 if is_hgg else 0
        
        for s_idx in range(slices_per_subject):
            # Create multi-modal slice with variation per slice
            base = np.random.uniform(0.2, 0.7)
            channels = []
            
            for mod_scale in [1.0, 0.8, 0.9, 1.1]:
                ch = np.zeros((H, W), dtype=np.float32)
                ch[brain_mask] = base * mod_scale + np.random.normal(0, 0.04, brain_mask.sum())
                
                # Add tumor with some slice-to-slice variation
                tcx = 120 + np.random.randint(-20, 20)
                tcy = 120 + np.random.randint(-20, 20)
                tr = np.random.randint(20, 40) if is_hgg else np.random.randint(8, 20)
                # Vary tumor size across slices
                tr = max(5, int(tr * (1 - abs(s_idx - slices_per_subject/2) / slices_per_subject)))
                
                tumor = ((np.arange(W) - tcx)**2 + (np.arange(H)[:, None] - tcy)**2) < tr**2
                tumor = tumor & brain_mask
                
                enhancement = np.random.uniform(1.2, 1.5) if is_hgg else np.random.uniform(1.05, 1.2)
                ch[tumor] *= enhancement
                channels.append(np.clip(ch, 0, 1))
            
            multi_modal = np.stack(channels, axis=-1)
            result = preprocess_slice(multi_modal, target_size)
            
            all_signals.append(result['signal_1d'])
            all_labels.append(label)
            all_metadata.append({'subject_id': f'Synthetic_{subj_idx:03d}', 'slice_no': s_idx})
    
    return np.array(all_signals), np.array(all_labels), all_metadata


print("✅ Dataset processing functions defined")

In [ ]:
# Process the dataset
print("\n" + "="*60)
print(" PROCESSING DATASET")
print("="*60)

if USE_SYNTHETIC:
    signals, labels, metadata = process_dataset_synthetic(
        n_subjects=MAX_SUBJECTS,
        slices_per_subject=SLICES_PER_SUBJECT,
        target_size=TARGET_SIZE
    )
else:
    signals, labels, metadata = process_dataset_nifti(
        data_dir=DATA_DIR,
        label_map=label_map,
        target_size=TARGET_SIZE,
        slices_per_subject=SLICES_PER_SUBJECT,
        max_subjects=MAX_SUBJECTS
    )

print(f"\n📊 Processing Results:")
print(f"   Total samples    : {len(signals)}")
print(f"   Signal shape     : {signals.shape}")
print(f"   Labels shape     : {labels.shape}")
print(f"   Malignant (HGG)  : {np.sum(labels == 1)}")
print(f"   Benign (LGG)     : {np.sum(labels == 0)}")
print(f"   Signal range     : [{signals.min():.4f}, {signals.max():.4f}]")
print(f"   Signal mean      : {signals.mean():.4f}")
print(f"   Signal std       : {signals.std():.4f}")

## 8. Statistical Analysis of 1D Signals

In [ ]:
# Separate signals by class
benign_idx = labels == 0
malignant_idx = labels == 1

benign_sigs = signals[benign_idx]
malignant_sigs = signals[malignant_idx]

# Compute statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Mean signal profiles
t = np.arange(TARGET_SIZE[0])
mean_b = np.mean(benign_sigs, axis=0)
std_b = np.std(benign_sigs, axis=0)
mean_m = np.mean(malignant_sigs, axis=0)
std_m = np.std(malignant_sigs, axis=0)

axes[0, 0].plot(t, mean_b, 'o-', color='seagreen', linewidth=2.5, label='Benign (mean)', markersize=8)
axes[0, 0].fill_between(t, mean_b - std_b, mean_b + std_b, alpha=0.2, color='seagreen')
axes[0, 0].plot(t, mean_m, 's-', color='crimson', linewidth=2.5, label='Malignant (mean)', markersize=8)
axes[0, 0].fill_between(t, mean_m - std_m, mean_m + std_m, alpha=0.2, color='crimson')
axes[0, 0].set_title('Mean Signal Profiles (±1 std)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Row Index (t)')
axes[0, 0].set_ylabel('Mean Intensity')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. Signal energy distribution
energy_b = np.sum(benign_sigs**2, axis=1)
energy_m = np.sum(malignant_sigs**2, axis=1)
axes[0, 1].hist(energy_b, bins=30, alpha=0.6, color='seagreen', label=f'Benign (n={len(energy_b)})', density=True)
axes[0, 1].hist(energy_m, bins=30, alpha=0.6, color='crimson', label=f'Malignant (n={len(energy_m)})', density=True)
axes[0, 1].set_title('Signal Energy Distribution', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Signal Energy (Σ s²)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. Per-index box plots
data_for_box = []
positions = []
colors_for_box = []
for i in range(TARGET_SIZE[0]):
    data_for_box.append(benign_sigs[:, i])
    data_for_box.append(malignant_sigs[:, i])
    positions.extend([i*3, i*3+1])

bp = axes[1, 0].boxplot(data_for_box, positions=positions, widths=0.8, patch_artist=True)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('seagreen' if i % 2 == 0 else 'crimson')
    patch.set_alpha(0.6)
axes[1, 0].set_xticks([i*3+0.5 for i in range(TARGET_SIZE[0])])
axes[1, 0].set_xticklabels([str(i) for i in range(TARGET_SIZE[0])])
axes[1, 0].set_title('Per-Index Distribution (Green=Benign, Red=Malignant)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Row Index')
axes[1, 0].set_ylabel('Intensity')
axes[1, 0].grid(True, alpha=0.3)

# 4. Correlation heatmap of signal indices
corr = np.corrcoef(signals.T)
im = axes[1, 1].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[1, 1], shrink=0.8)
axes[1, 1].set_title('Signal Index Correlation Matrix', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Row Index')
axes[1, 1].set_ylabel('Row Index')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'signal_statistics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Preprocessed Data

Save the processed 1D signals and labels for use in subsequent notebooks (Differential Equation Modeling and CNN Classification).

In [ ]:
# Save preprocessed signals and labels
output_file = os.path.join(OUTPUT_DIR, 'preprocessed_signals.npz')

np.savez_compressed(
    output_file,
    signals=signals,
    labels=labels,
    target_size=np.array(TARGET_SIZE),
    signal_length=np.array([signals.shape[1]])
)

# Also save metadata
meta_df = pd.DataFrame(metadata)
meta_df['label'] = labels
meta_df['label_name'] = meta_df['label'].map({0: 'Benign (LGG)', 1: 'Malignant (HGG)'})
meta_df.to_csv(os.path.join(OUTPUT_DIR, 'preprocessed_metadata.csv'), index=False)

print("✅ Preprocessed data saved successfully!")
print(f"   Signals file : {output_file} ({os.path.getsize(output_file)/1024:.1f} KB)")
print(f"   Metadata file: {os.path.join(OUTPUT_DIR, 'preprocessed_metadata.csv')}")
print(f"\n   Signals shape: {signals.shape}")
print(f"   Labels shape : {labels.shape}")
print(f"   Benign count : {np.sum(labels == 0)}")
print(f"   Malignant    : {np.sum(labels == 1)}")

---

## ✅ Summary

In this notebook, we implemented the complete preprocessing pipeline:

| Step | Input | Output | Description |
|------|-------|--------|-------------|
| 1. Grayscale | (240, 240, 4) | (240, 240) | Weighted mean across 4 MRI modalities |
| 2. Resize | (240, 240) | (10, 10) | Bilinear interpolation with anti-aliasing |
| 3. Row Mean | (10, 10) | (10,) | Mean intensity per row → 1D signal |

### Key Observations
- The 1D signals capture the **vertical intensity profile** of brain slices
- Malignant tumors tend to produce signals with different amplitude and variation patterns
- The aggressive downscaling to 10×10 preserves global patterns while reducing dimensionality

### ➡️ Next: [Notebook 03 - Differential Equation Modeling](./03_Differential_Equation_Modeling.ipynb)

The next notebook will model these 1D signals using:
- **Ordinary Differential Equations (ODE)** with Euler's method and RK4
- **Integro-Differential Equations (IDE)** for enhanced feature extraction